In [6]:
"""
PACR-DETR for SAV Dataset - Bounding Box Detection Only
=========================================================
This model detects student bounding boxes in classroom videos.
Action classification is handled separately by TST (Temporal Segment Transformer).

Features:
- Multi-scale fusion (P3 + P4 + P5)
- Position-Aware Channel Residual (PACR) attention
- Improved transformer with 4 encoder layers
- Cosine annealing with warm restarts
- Strong data augmentation
- Complete training, resume, inference, and evaluation pipeline

Author: Srinithi
Date: March 2026
"""

import os
import math
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Any
from collections import defaultdict
import json
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from scipy.optimize import linear_sum_assignment

warnings.filterwarnings('ignore')


# ============================================================================
# CONFIGURATION
# ============================================================================
@dataclass
class Config:
    """Configuration for PACR-DETR Detection Model"""
    
    # Data paths (Kaggle)
    data_root: str = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/sav-dataset/SAV/data/sav/frames"
    train_ann: str = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/sav-dataset/SAV/annotations/train.csv"
    val_ann: str = "/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/sav-dataset/SAV/annotations/val.csv"
    output_dir: str = "/kaggle/working/checkpoints"
    
    # Model Architecture
    hidden_dim: int = 256
    num_queries: int = 100
    num_encoder_layers: int = 4      # Increased from 2
    num_decoder_layers: int = 6
    nheads: int = 8
    dropout: float = 0.1
    
    # Training
    epochs: int = 10
    batch_size: int = 16
    lr: float = 2e-4                 # Increased
    lr_backbone: float = 2e-5        # Increased
    weight_decay: float = 1e-4
    clip_grad: float = 0.1
    warmup_epochs: int = 3
    
    # Loss weights
    obj_weight: float = 1.0
    bbox_weight: float = 5.0
    giou_weight: float = 2.0
    
    # Data
    img_size: int = 640
    num_frames: int = 90
    keyframe: int = 45
    propagation_range: int = 15
    
    # Evaluation
    conf_threshold: float = 0.5
    iou_thresholds: Tuple[float, ...] = (0.5, 0.75)  # AP@50, AP@75
    
    # Device
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    num_workers: int = 2
    
    # Checkpoint
    save_every: int = 2
    
    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)


# ============================================================================
# PACR MODULE - Position-Aware Channel Residual
# ============================================================================
class CoordinateAttention(nn.Module):
    """Coordinate Attention for position-aware features"""
    
    def __init__(self, in_channels: int, reduction: int = 32):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        
        mid_channels = max(8, in_channels // reduction)
        
        self.conv1 = nn.Conv2d(in_channels, mid_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(mid_channels)
        self.act = nn.ReLU(inplace=True)
        
        self.conv_h = nn.Conv2d(mid_channels, in_channels, kernel_size=1, bias=False)
        self.conv_w = nn.Conv2d(mid_channels, in_channels, kernel_size=1, bias=False)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        
        y_h = self.act(self.bn1(self.conv1(x_h)))
        y_w = self.act(self.bn1(self.conv1(x_w)))
        
        a_h = self.conv_h(y_h).sigmoid()
        a_w = self.conv_w(y_w).permute(0, 1, 3, 2).sigmoid()
        
        return x * a_h * a_w


class PACR(nn.Module):
    """Position-Aware Channel Residual Module"""
    
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.coord_att = CoordinateAttention(channels, reduction)
        
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        mid = max(8, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False)
        )
        
        self.gamma = nn.Parameter(torch.zeros(1))
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        
        # Position attention
        pos_att = self.coord_att(x)
        
        # Channel attention
        avg_out = self.fc(self.avg_pool(x).view(B, C))
        max_out = self.fc(self.max_pool(x).view(B, C))
        ch_att = torch.sigmoid(avg_out + max_out).view(B, C, 1, 1)
        
        out = pos_att * ch_att
        
        return x + self.gamma * out


class PACRBottleneck(nn.Module):
    """ResNet Bottleneck with PACR"""
    expansion = 4
    
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1, downsample: nn.Module = None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv3 = nn.Conv2d(out_ch, out_ch * self.expansion, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.pacr = PACR(out_ch * self.expansion)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = self.pacr(out)
        
        if self.downsample is not None:
            identity = self.downsample(x)
            
        out = out + identity
        return self.relu(out)


# ============================================================================
# BACKBONE - PACRNet (ResNet50 with PACR)
# ============================================================================
class PACRNet(nn.Module):
    """ResNet50 backbone with PACR modules"""
    
    def __init__(self, pretrained: bool = True):
        super().__init__()
        
        self.conv1 = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        
        self.in_channels = 64
        self.layer1 = self._make_layer(64, 3, stride=1)
        self.layer2 = self._make_layer(128, 4, stride=2)
        self.layer3 = self._make_layer(256, 6, stride=2)
        self.layer4 = self._make_layer(512, 3, stride=2)
        
        self.out_channels = [512, 1024, 2048]  # C3, C4, C5
        
        self._init_weights()
        
        if pretrained:
            self._load_pretrained()
            
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            
    def _make_layer(self, channels: int, blocks: int, stride: int) -> nn.Sequential:
        downsample = None
        if stride != 1 or self.in_channels != channels * PACRBottleneck.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, channels * PACRBottleneck.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(channels * PACRBottleneck.expansion)
            )
        
        layers = [PACRBottleneck(self.in_channels, channels, stride, downsample)]
        self.in_channels = channels * PACRBottleneck.expansion
        for _ in range(1, blocks):
            layers.append(PACRBottleneck(self.in_channels, channels))
            
        return nn.Sequential(*layers)
    
    def _load_pretrained(self):
        """Load pretrained ResNet50 weights"""
        try:
            resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        except:
            resnet = models.resnet50(pretrained=True)
        
        # Copy conv1 and bn1
        self.conv1.weight.data.copy_(resnet.conv1.weight.data)
        self.bn1.weight.data.copy_(resnet.bn1.weight.data)
        self.bn1.bias.data.copy_(resnet.bn1.bias.data)
        self.bn1.running_mean.copy_(resnet.bn1.running_mean)
        self.bn1.running_var.copy_(resnet.bn1.running_var)
        
        # Copy layer weights
        for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
            src_layer = getattr(resnet, layer_name)
            dst_layer = getattr(self, layer_name)
            
            for i, (src_block, dst_block) in enumerate(zip(src_layer, dst_layer)):
                dst_block.conv1.weight.data.copy_(src_block.conv1.weight.data)
                dst_block.conv2.weight.data.copy_(src_block.conv2.weight.data)
                dst_block.conv3.weight.data.copy_(src_block.conv3.weight.data)
                
                for bn_name in ['bn1', 'bn2', 'bn3']:
                    src_bn = getattr(src_block, bn_name)
                    dst_bn = getattr(dst_block, bn_name)
                    dst_bn.weight.data.copy_(src_bn.weight.data)
                    dst_bn.bias.data.copy_(src_bn.bias.data)
                    dst_bn.running_mean.copy_(src_bn.running_mean)
                    dst_bn.running_var.copy_(src_bn.running_var)
                
                if dst_block.downsample is not None and src_block.downsample is not None:
                    dst_block.downsample[0].weight.data.copy_(src_block.downsample[0].weight.data)
                    dst_block.downsample[1].weight.data.copy_(src_block.downsample[1].weight.data)
                    dst_block.downsample[1].bias.data.copy_(src_block.downsample[1].bias.data)
                    dst_block.downsample[1].running_mean.copy_(src_block.downsample[1].running_mean)
                    dst_block.downsample[1].running_var.copy_(src_block.downsample[1].running_var)
                    
        print("✓ Loaded pretrained ResNet50 weights")
        
    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        
        x = self.layer1(x)
        c3 = self.layer2(x)   # 1/8 resolution
        c4 = self.layer3(c3)  # 1/16 resolution
        c5 = self.layer4(c4)  # 1/32 resolution
        
        return [c3, c4, c5]


# ============================================================================
# NECK - Feature Pyramid Network
# ============================================================================
class FPN(nn.Module):
    """Feature Pyramid Network"""
    
    def __init__(self, in_channels: List[int], out_channels: int = 256):
        super().__init__()
        
        self.lateral_convs = nn.ModuleList()
        self.fpn_convs = nn.ModuleList()
        
        for in_ch in in_channels:
            self.lateral_convs.append(
                nn.Sequential(
                    nn.Conv2d(in_ch, out_channels, 1, bias=False),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU(inplace=True)
                )
            )
            self.fpn_convs.append(
                nn.Sequential(
                    nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU(inplace=True)
                )
            )
        
    def forward(self, features: List[torch.Tensor]) -> List[torch.Tensor]:
        # Lateral connections
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]
        
        # Top-down pathway
        for i in range(len(laterals) - 1, 0, -1):
            laterals[i-1] = laterals[i-1] + F.interpolate(
                laterals[i], size=laterals[i-1].shape[-2:], mode='nearest'
            )
        
        # Output convolutions
        outputs = [conv(lat) for conv, lat in zip(self.fpn_convs, laterals)]
        return outputs


# ============================================================================
# MULTI-SCALE FUSION MODULE
# ============================================================================
class MultiScaleFusion(nn.Module):
    """Fuse P3, P4, P5 features into single representation"""
    
    def __init__(self, in_channels: int = 256, out_channels: int = 256):
        super().__init__()
        
        # Separate projections for each scale
        self.proj_p3 = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.proj_p4 = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.proj_p5 = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        
        # Fusion layers
        self.fusion = nn.Sequential(
            nn.Conv2d(out_channels * 3, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        
        # Learnable scale weights
        self.scale_weights = nn.Parameter(torch.ones(3) / 3)
        
        self._init_weights()
        
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, p3: torch.Tensor, p4: torch.Tensor, p5: torch.Tensor) -> torch.Tensor:
        """
        Args:
            p3: (B, C, H/8, W/8)   - 80x80 for 640 input
            p4: (B, C, H/16, W/16) - 40x40
            p5: (B, C, H/32, W/32) - 20x20
        Returns:
            fused: (B, C, H/32, W/32) - 20x20
        """
        target_size = p5.shape[-2:]  # (20, 20)
        
        # Project each scale
        p3_proj = self.proj_p3(p3)
        p4_proj = self.proj_p4(p4)
        p5_proj = self.proj_p5(p5)
        
        # Downsample to target size
        p3_down = F.interpolate(p3_proj, size=target_size, mode='bilinear', align_corners=False)
        p4_down = F.interpolate(p4_proj, size=target_size, mode='bilinear', align_corners=False)
        
        # Normalize scale weights
        weights = F.softmax(self.scale_weights, dim=0)
        
        # Weighted sum + concatenation fusion
        weighted_sum = weights[0] * p3_down + weights[1] * p4_down + weights[2] * p5_proj
        concat_features = torch.cat([p3_down, p4_down, p5_proj], dim=1)
        
        # Fuse
        fused = self.fusion(concat_features)
        
        # Residual with weighted sum
        fused = fused + weighted_sum
        
        return fused


# ============================================================================
# TRANSFORMER
# ============================================================================
class PositionalEncoding2D(nn.Module):
    """2D Positional Encoding for spatial features"""
    
    def __init__(self, d_model: int = 256, temperature: float = 10000.0):
        super().__init__()
        self.d_model = d_model
        self.temperature = temperature
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        device = x.device
        
        y_embed = torch.arange(H, device=device).float().unsqueeze(1).repeat(1, W)
        x_embed = torch.arange(W, device=device).float().unsqueeze(0).repeat(H, 1)
        
        y_embed = y_embed / (H + 1e-6)
        x_embed = x_embed / (W + 1e-6)
        
        dim_t = torch.arange(self.d_model // 2, device=device).float()
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.d_model)
        
        pos_x = x_embed.unsqueeze(-1) / dim_t
        pos_y = y_embed.unsqueeze(-1) / dim_t
        
        pos_x = torch.stack([pos_x[:, :, 0::2].sin(), pos_x[:, :, 1::2].cos()], dim=-1).flatten(-2)
        pos_y = torch.stack([pos_y[:, :, 0::2].sin(), pos_y[:, :, 1::2].cos()], dim=-1).flatten(-2)
        
        pos = torch.cat([pos_y, pos_x], dim=-1)
        pos = pos.permute(2, 0, 1).unsqueeze(0).repeat(B, 1, 1, 1)
        
        return pos


class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, nhead: int, dim_feedforward: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
        self.activation = nn.GELU()  # Changed to GELU
        
    def forward(self, src: torch.Tensor, pos: torch.Tensor) -> torch.Tensor:
        q = k = src + pos
        src2, _ = self.self_attn(q, k, src)
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        
        return src


class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model: int, nhead: int, dim_feedforward: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
        self.activation = nn.GELU()
        
    def forward(self, tgt: torch.Tensor, memory: torch.Tensor, 
                query_pos: torch.Tensor, pos: torch.Tensor) -> torch.Tensor:
        # Self attention
        q = k = tgt + query_pos
        tgt2, _ = self.self_attn(q, k, tgt)
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        
        # Cross attention
        tgt2, _ = self.cross_attn(tgt + query_pos, memory + pos, memory)
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        
        # FFN
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        
        return tgt


class Transformer(nn.Module):
    def __init__(self, d_model: int = 256, nhead: int = 8, 
                 num_encoder_layers: int = 4, num_decoder_layers: int = 6,
                 dim_feedforward: int = 2048, dropout: float = 0.1):
        super().__init__()
        
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_encoder_layers)
        ])
        
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_decoder_layers)
        ])
        
        self.d_model = d_model
        
    def forward(self, src: torch.Tensor, pos: torch.Tensor, 
                query_embed: torch.Tensor) -> torch.Tensor:
        B, C, H, W = src.shape
        
        # Flatten spatial dimensions
        src = src.flatten(2).permute(0, 2, 1)  # B, H*W, C
        pos = pos.flatten(2).permute(0, 2, 1)  # B, H*W, C
        
        # Encoder
        memory = src
        for layer in self.encoder_layers:
            memory = layer(memory, pos)
        
        # Decoder
        tgt = query_embed.unsqueeze(0).repeat(B, 1, 1)
        query_pos = query_embed.unsqueeze(0).repeat(B, 1, 1)
        
        for layer in self.decoder_layers:
            tgt = layer(tgt, memory, query_pos, pos)
        
        return tgt


# ============================================================================
# PACR-DETR MODEL (Detection Only)
# ============================================================================
class PACRDETRDetector(nn.Module):
    """
    PACR-DETR for Bounding Box Detection Only
    
    This model outputs:
    - pred_boxes: (B, num_queries, 4) in cxcywh format, normalized [0,1]
    - pred_obj: (B, num_queries, 1) objectness scores
    """
    
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.num_queries = config.num_queries
        self.hidden_dim = config.hidden_dim
        
        # Backbone
        self.backbone = PACRNet(pretrained=True)
        
        # Neck
        self.neck = FPN(self.backbone.out_channels, config.hidden_dim)
        
        # Multi-scale fusion
        self.multi_scale_fusion = MultiScaleFusion(config.hidden_dim, config.hidden_dim)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding2D(config.hidden_dim)
        
        # Transformer
        self.transformer = Transformer(
            d_model=config.hidden_dim,
            nhead=config.nheads,
            num_encoder_layers=config.num_encoder_layers,
            num_decoder_layers=config.num_decoder_layers,
            dim_feedforward=config.hidden_dim * 4,
            dropout=config.dropout
        )
        
        # Query embeddings
        self.query_embed = nn.Embedding(config.num_queries, config.hidden_dim)
        
        # Objectness head (is this a person?)
        self.obj_head = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, 1)
        )
        
        # Bounding box head
        self.bbox_head = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(config.hidden_dim, config.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(config.hidden_dim, 4)
        )
        
        self._init_weights()
        
    def _init_weights(self):
        # Initialize heads
        for head in [self.obj_head, self.bbox_head]:
            for layer in head:
                if isinstance(layer, nn.Linear):
                    nn.init.xavier_uniform_(layer.weight)
                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)
        
        # Initialize objectness with prior
        prior_prob = 0.01
        bias_value = -math.log((1 - prior_prob) / prior_prob)
        nn.init.constant_(self.obj_head[-1].bias, bias_value)
        
        # Initialize query embeddings
        nn.init.normal_(self.query_embed.weight, std=0.01)
        
    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Args:
            x: Input images (B, 3, H, W)
        Returns:
            dict with:
                - pred_boxes: (B, num_queries, 4) normalized cxcywh
                - pred_obj: (B, num_queries, 1) objectness logits
        """
        # Backbone
        features = self.backbone(x)  # [C3, C4, C5]
        
        # FPN
        fpn_features = self.neck(features)  # [P3, P4, P5]
        P3, P4, P5 = fpn_features
        
        # Multi-scale fusion
        src = self.multi_scale_fusion(P3, P4, P5)  # (B, C, 20, 20)
        
        # Positional encoding
        pos = self.pos_encoder(src)
        
        # Transformer
        hs = self.transformer(src, pos, self.query_embed.weight)  # (B, num_queries, hidden_dim)
        
        # Prediction heads
        pred_obj = self.obj_head(hs)          # (B, num_queries, 1)
        pred_boxes = self.bbox_head(hs).sigmoid()  # (B, num_queries, 4)
        
        return {
            'pred_boxes': pred_boxes,
            'pred_obj': pred_obj
        }


# ============================================================================
# DATASET
# ============================================================================
class SAVDetectionDataset(Dataset):
    """SAV Dataset for Person Detection (boxes only, no action labels)"""
    
    def __init__(self, config: Config, ann_file: str, is_train: bool = True):
        super().__init__()
        self.config = config
        self.is_train = is_train
        self.img_size = config.img_size
        self.data_root = Path(config.data_root)
        
        # Load annotations
        self.annotations, self.clip_to_persons = self._load_annotations(ann_file)
        
        # Get available clips
        self.available_clips = self._get_available_clips()
        
        # Build samples
        self.samples = self._build_samples()
        
        # Transforms
        self.transform = self._build_transforms()
        
    def _load_annotations(self, ann_file: str) -> Tuple[Dict, Dict]:
        """Load annotations - extract unique person boxes per clip"""
        annotations = defaultdict(dict)  # clip_name -> {person_id: bbox}
        clip_to_persons = defaultdict(dict)
        
        try:
            df = pd.read_csv(ann_file, header=None)
            
            if df.shape[1] >= 8:
                df.columns = ['clip_name', 'const', 'x1', 'y1', 'x2', 'y2', 'action_id', 'person_id'] + \
                            [f'col_{i}' for i in range(8, df.shape[1])]
            else:
                return dict(annotations), dict(clip_to_persons)
            
            for _, row in df.iterrows():
                clip_name = str(row['clip_name']).strip()
                
                try:
                    person_id = int(row['person_id'])
                    x1 = float(row['x1'])
                    y1 = float(row['y1'])
                    x2 = float(row['x2'])
                    y2 = float(row['y2'])
                except (ValueError, TypeError):
                    continue
                
                # Skip invalid boxes
                if x2 <= x1 or y2 <= y1:
                    continue
                
                bbox = [x1, y1, x2, y2]
                
                # Store unique person boxes (first occurrence)
                if person_id not in annotations[clip_name]:
                    annotations[clip_name][person_id] = bbox
                    clip_to_persons[clip_name][person_id] = bbox
                    
        except Exception as e:
            print(f"Error loading annotations: {e}")
            
        return dict(annotations), dict(clip_to_persons)
    
    def _get_available_clips(self) -> List[str]:
        """Get clips that exist in both annotations and directory"""
        available = []
        
        if not self.data_root.exists():
            return available
        
        dir_names = set(d.name for d in self.data_root.iterdir() if d.is_dir())
        
        for clip_name in self.annotations.keys():
            if clip_name in dir_names:
                available.append(clip_name)
        
        return available
    
    def _build_samples(self) -> List[Tuple[str, int]]:
        """Build (clip_name, frame_idx) samples"""
        samples = []
        
        for clip_name in self.available_clips:
            if self.is_train:
                start = max(0, self.config.keyframe - self.config.propagation_range)
                end = min(self.config.num_frames, self.config.keyframe + self.config.propagation_range + 1)
                for frame_idx in range(start, end):
                    samples.append((clip_name, frame_idx))
            else:
                samples.append((clip_name, self.config.keyframe))
                
        return samples
    
    def _build_transforms(self):
        if self.is_train:
            return transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
                transforms.RandomGrayscale(p=0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
        else:
            return transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
    
    def _find_frame_path(self, clip_name: str, frame_idx: int) -> Optional[Path]:
        """Find frame file"""
        clip_dir = self.data_root / clip_name
        
        if not clip_dir.exists():
            return None
        
        patterns = [
            f"img_{frame_idx:06d}.jpg",
            f"{frame_idx:06d}.jpg",
            f"{frame_idx:05d}.jpg",
            f"frame_{frame_idx:06d}.jpg",
        ]
        
        for pattern in patterns:
            path = clip_dir / pattern
            if path.exists():
                return path
        
        # Fallback: list directory
        try:
            files = sorted([f for f in clip_dir.iterdir() if f.suffix.lower() in ['.jpg', '.png']])
            if 0 <= frame_idx < len(files):
                return files[frame_idx]
        except:
            pass
        
        return None
    
    def __len__(self) -> int:
        return len(self.samples)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict]:
        clip_name, frame_idx = self.samples[idx]
        
        # Load image
        frame_path = self._find_frame_path(clip_name, frame_idx)
        if frame_path is None:
            frame_path = self._find_frame_path(clip_name, self.config.keyframe)
        
        if frame_path is None:
            return self._get_dummy_item(clip_name, frame_idx)
        
        try:
            image = Image.open(frame_path).convert('RGB')
        except:
            return self._get_dummy_item(clip_name, frame_idx)
        
        orig_w, orig_h = image.size
        
        # Get person boxes
        clip_anns = self.annotations.get(clip_name, {})
        boxes = []
        
        # Track if we flip for bbox adjustment
        do_flip = self.is_train and random.random() < 0.5
        
        for person_id, bbox in clip_anns.items():
            x1, y1, x2, y2 = bbox
            
            # Check if normalized
            if all(0 <= c <= 1 for c in [x1, y1, x2, y2]):
                pass  # Already normalized
            else:
                x1 = max(0, min(orig_w, x1)) / orig_w
                y1 = max(0, min(orig_h, y1)) / orig_h
                x2 = max(0, min(orig_w, x2)) / orig_w
                y2 = max(0, min(orig_h, y2)) / orig_h
            
            if x2 <= x1 or y2 <= y1:
                continue
            
            # Handle horizontal flip
            if do_flip:
                x1_new = 1.0 - x2
                x2_new = 1.0 - x1
                x1, x2 = x1_new, x2_new
            
            # Convert xyxy to cxcywh
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            w = x2 - x1
            h = y2 - y1
            
            # Clamp
            cx = max(0.001, min(0.999, cx))
            cy = max(0.001, min(0.999, cy))
            w = max(0.001, min(0.999, w))
            h = max(0.001, min(0.999, h))
            
            boxes.append([cx, cy, w, h])
        
        # Resize image
        image = image.resize((self.img_size, self.img_size), Image.BILINEAR)
        
        # Apply flip to image if needed
        if do_flip:
            image = image.transpose(Image.FLIP_LEFT_RIGHT)
        
        # Apply other transforms (color jitter, etc.)
        if self.is_train:
            # Apply color transforms only (flip already done)
            color_transform = transforms.Compose([
                transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.15),
                transforms.RandomGrayscale(p=0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
            image = color_transform(image)
        else:
            image = self.transform(image)
        
        # Convert to tensors
        if len(boxes) > 0:
            boxes = torch.tensor(boxes, dtype=torch.float32)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
        
        target = {
            'boxes': boxes,
            'clip_name': clip_name,
            'frame_idx': frame_idx,
            'num_objects': len(boxes)
        }
        
        return image, target
    
    def _get_dummy_item(self, clip_name: str, frame_idx: int) -> Tuple[torch.Tensor, Dict]:
        """Return dummy item for missing frames"""
        image = torch.randn(3, self.img_size, self.img_size)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image * std) + mean
        
        target = {
            'boxes': torch.zeros((0, 4), dtype=torch.float32),
            'clip_name': clip_name,
            'frame_idx': frame_idx,
            'num_objects': 0
        }
        return image, target


def collate_fn(batch):
    """Custom collate function"""
    images = torch.stack([item[0] for item in batch])
    targets = [item[1] for item in batch]
    return images, targets


# ============================================================================
# LOSS FUNCTIONS
# ============================================================================
class HungarianMatcher(nn.Module):
    """Hungarian Matcher for detection"""
    
    def __init__(self, cost_obj: float = 1.0, cost_bbox: float = 5.0, cost_giou: float = 2.0):
        super().__init__()
        self.cost_obj = cost_obj
        self.cost_bbox = cost_bbox
        self.cost_giou = cost_giou
        
    @torch.no_grad()
    def forward(self, outputs: Dict[str, torch.Tensor], targets: List[Dict]) -> List[Tuple]:
        B = outputs['pred_boxes'].shape[0]
        num_queries = outputs['pred_boxes'].shape[1]
        device = outputs['pred_boxes'].device
        
        indices = []
        
        for b in range(B):
            tgt_boxes = targets[b]['boxes'].to(device)
            num_tgt = tgt_boxes.shape[0]
            
            if num_tgt == 0:
                indices.append((
                    torch.tensor([], dtype=torch.int64, device=device),
                    torch.tensor([], dtype=torch.int64, device=device)
                ))
                continue
            
            pred_obj = outputs['pred_obj'][b].sigmoid().squeeze(-1)  # Q
            pred_boxes = outputs['pred_boxes'][b]  # Q, 4
            
            # Objectness cost (prefer high objectness)
            cost_obj = -pred_obj.unsqueeze(1).repeat(1, num_tgt)  # Q, num_tgt
            
            # L1 cost
            cost_bbox = torch.cdist(pred_boxes, tgt_boxes, p=1)  # Q, num_tgt
            
            # GIoU cost
            cost_giou = -self._batch_giou(pred_boxes, tgt_boxes)  # Q, num_tgt
            
            # Total cost
            C = (self.cost_obj * cost_obj + 
                 self.cost_bbox * cost_bbox + 
                 self.cost_giou * cost_giou)
            
            # Hungarian matching
            C_cpu = C.cpu().numpy()
            row_ind, col_ind = linear_sum_assignment(C_cpu)
            
            indices.append((
                torch.tensor(row_ind, dtype=torch.int64, device=device),
                torch.tensor(col_ind, dtype=torch.int64, device=device)
            ))
        
        return indices
    
    def _batch_giou(self, boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
        """Compute GIoU between all pairs"""
        b1 = self._cxcywh_to_xyxy(boxes1)
        b2 = self._cxcywh_to_xyxy(boxes2)
        
        area1 = (b1[:, 2] - b1[:, 0]).clamp(min=0) * (b1[:, 3] - b1[:, 1]).clamp(min=0)
        area2 = (b2[:, 2] - b2[:, 0]).clamp(min=0) * (b2[:, 3] - b2[:, 1]).clamp(min=0)
        
        inter_x1 = torch.max(b1[:, None, 0], b2[None, :, 0])
        inter_y1 = torch.max(b1[:, None, 1], b2[None, :, 1])
        inter_x2 = torch.min(b1[:, None, 2], b2[None, :, 2])
        inter_y2 = torch.min(b1[:, None, 3], b2[None, :, 3])
        
        inter_area = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
        union = area1[:, None] + area2[None, :] - inter_area
        iou = inter_area / (union + 1e-6)
        
        enc_x1 = torch.min(b1[:, None, 0], b2[None, :, 0])
        enc_y1 = torch.min(b1[:, None, 1], b2[None, :, 1])
        enc_x2 = torch.max(b1[:, None, 2], b2[None, :, 2])
        enc_y2 = torch.max(b1[:, None, 3], b2[None, :, 3])
        
        enc_area = (enc_x2 - enc_x1) * (enc_y2 - enc_y1)
        giou = iou - (enc_area - union) / (enc_area + 1e-6)
        
        return giou
    
    def _cxcywh_to_xyxy(self, boxes: torch.Tensor) -> torch.Tensor:
        cx, cy, w, h = boxes.unbind(-1)
        return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)


class DetectionLoss(nn.Module):
    """Loss function for detection (objectness + bbox)"""
    
    def __init__(self, config: Config):
        super().__init__()
        self.config = config
        self.num_queries = config.num_queries
        
        self.matcher = HungarianMatcher(
            cost_obj=config.obj_weight,
            cost_bbox=config.bbox_weight,
            cost_giou=config.giou_weight
        )
        
        self.obj_weight = config.obj_weight
        self.bbox_weight = config.bbox_weight
        self.giou_weight = config.giou_weight
        
        self.eos_coef = 0.1  # Weight for no-object
        
    def forward(self, outputs: Dict[str, torch.Tensor], 
                targets: List[Dict]) -> Dict[str, torch.Tensor]:
        device = outputs['pred_boxes'].device
        
        # Move targets to device
        targets = [{
            'boxes': t['boxes'].to(device),
            'num_objects': t['num_objects']
        } for t in targets]
        
        # Hungarian matching
        indices = self.matcher(outputs, targets)
        
        num_matched = sum(len(idx[0]) for idx in indices)
        
        # Compute losses
        loss_obj = self._loss_objectness(outputs, targets, indices)
        loss_bbox = self._loss_bbox_l1(outputs, targets, indices)
        loss_giou = self._loss_giou(outputs, targets, indices)
        
        losses = {
            'loss_obj': self.obj_weight * loss_obj,
            'loss_bbox': self.bbox_weight * loss_bbox,
            'loss_giou': self.giou_weight * loss_giou,
            'num_matched': torch.tensor(num_matched, dtype=torch.float32, device=device)
        }
        
        losses['loss_total'] = losses['loss_obj'] + losses['loss_bbox'] + losses['loss_giou']
        
        return losses
    
    def _loss_objectness(self, outputs: Dict, targets: List[Dict],
                         indices: List[Tuple]) -> torch.Tensor:
        """Binary objectness loss"""
        pred_obj = outputs['pred_obj']  # B, Q, 1
        B, Q = pred_obj.shape[:2]
        device = pred_obj.device
        
        target_obj = torch.zeros(B, Q, 1, device=device)
        
        for b, (src_idx, _) in enumerate(indices):
            if len(src_idx) > 0:
                target_obj[b, src_idx] = 1.0
        
        # Focal-like weighting
        pred_prob = pred_obj.sigmoid()
        
        # Weighted BCE
        pos_weight = torch.ones_like(target_obj)
        pos_weight[target_obj == 0] = self.eos_coef
        
        loss = F.binary_cross_entropy_with_logits(pred_obj, target_obj, weight=pos_weight)
        
        return loss
    
    def _loss_bbox_l1(self, outputs: Dict, targets: List[Dict],
                      indices: List[Tuple]) -> torch.Tensor:
        """L1 loss for matched boxes"""
        device = outputs['pred_boxes'].device
        
        src_boxes = []
        tgt_boxes = []
        
        for b, (src_idx, tgt_idx) in enumerate(indices):
            if len(src_idx) > 0:
                src_boxes.append(outputs['pred_boxes'][b, src_idx])
                tgt_boxes.append(targets[b]['boxes'][tgt_idx])
        
        if len(src_boxes) == 0:
            return torch.tensor(0.0, device=device, requires_grad=True)
        
        src_boxes = torch.cat(src_boxes, dim=0)
        tgt_boxes = torch.cat(tgt_boxes, dim=0)
        
        loss = F.l1_loss(src_boxes, tgt_boxes, reduction='mean')
        
        return loss
    
    def _loss_giou(self, outputs: Dict, targets: List[Dict],
                   indices: List[Tuple]) -> torch.Tensor:
        """GIoU loss for matched boxes"""
        device = outputs['pred_boxes'].device
        
        src_boxes = []
        tgt_boxes = []
        
        for b, (src_idx, tgt_idx) in enumerate(indices):
            if len(src_idx) > 0:
                src_boxes.append(outputs['pred_boxes'][b, src_idx])
                tgt_boxes.append(targets[b]['boxes'][tgt_idx])
        
        if len(src_boxes) == 0:
            return torch.tensor(0.0, device=device, requires_grad=True)
        
        src_boxes = torch.cat(src_boxes, dim=0)
        tgt_boxes = torch.cat(tgt_boxes, dim=0)
        
        src_xyxy = self._cxcywh_to_xyxy(src_boxes)
        tgt_xyxy = self._cxcywh_to_xyxy(tgt_boxes)
        
        giou = self._compute_giou(src_xyxy, tgt_xyxy)
        loss = (1 - giou).mean()
        
        return loss
    
    def _compute_giou(self, boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
        """Compute element-wise GIoU"""
        area1 = (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0) * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
        area2 = (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0) * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)
        
        inter_x1 = torch.max(boxes1[:, 0], boxes2[:, 0])
        inter_y1 = torch.max(boxes1[:, 1], boxes2[:, 1])
        inter_x2 = torch.min(boxes1[:, 2], boxes2[:, 2])
        inter_y2 = torch.min(boxes1[:, 3], boxes2[:, 3])
        
        inter_area = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
        union = area1 + area2 - inter_area
        iou = inter_area / (union + 1e-6)
        
        enc_x1 = torch.min(boxes1[:, 0], boxes2[:, 0])
        enc_y1 = torch.min(boxes1[:, 1], boxes2[:, 1])
        enc_x2 = torch.max(boxes1[:, 2], boxes2[:, 2])
        enc_y2 = torch.max(boxes1[:, 3], boxes2[:, 3])
        
        enc_area = (enc_x2 - enc_x1) * (enc_y2 - enc_y1)
        giou = iou - (enc_area - union) / (enc_area + 1e-6)
        
        return giou
    
    def _cxcywh_to_xyxy(self, boxes: torch.Tensor) -> torch.Tensor:
        cx, cy, w, h = boxes.unbind(-1)
        return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)


# ============================================================================
# EVALUATION METRICS
# ============================================================================
class DetectionEvaluator:
    """
    Evaluate detection performance with:
    - AP@IoU (Average Precision at different IoU thresholds)
    - AR (Average Recall)
    - Precision, Recall at conf_threshold
    """
    
    def __init__(self, config: Config):
        self.config = config
        self.conf_threshold = config.conf_threshold
        self.iou_thresholds = config.iou_thresholds
        self.reset()
        
    def reset(self):
        self.all_predictions = []
        self.all_targets = []
        
    @torch.no_grad()
    def update(self, outputs: Dict[str, torch.Tensor], targets: List[Dict]):
        """Add batch predictions and targets"""
        pred_obj = outputs['pred_obj'].sigmoid().cpu()
        pred_boxes = outputs['pred_boxes'].cpu()
        
        B = pred_boxes.shape[0]
        
        for b in range(B):
            obj_scores = pred_obj[b].squeeze(-1)
            
            # Filter by confidence
            valid_mask = obj_scores > self.conf_threshold
            
            if valid_mask.sum() > 0:
                valid_boxes = pred_boxes[b][valid_mask]
                valid_scores = obj_scores[valid_mask]
            else:
                valid_boxes = torch.zeros(0, 4)
                valid_scores = torch.zeros(0)
            
            tgt_boxes = targets[b]['boxes'].cpu()
            
            self.all_predictions.append({
                'boxes': valid_boxes,
                'scores': valid_scores
            })
            
            self.all_targets.append({
                'boxes': tgt_boxes
            })
    
    def compute(self) -> Dict[str, float]:
        """Compute all metrics"""
        metrics = {}
        
        # AP at different IoU thresholds
        for iou_thresh in self.iou_thresholds:
            ap = self._compute_ap(iou_thresh)
            metrics[f'AP@{int(iou_thresh*100)}'] = ap
        
        # Mean AP
        metrics['mAP'] = np.mean([metrics[f'AP@{int(t*100)}'] for t in self.iou_thresholds])
        
        # Precision, Recall at default threshold
        prec, rec = self._compute_precision_recall(self.iou_thresholds[0])
        metrics['precision'] = prec
        metrics['recall'] = rec
        metrics['f1'] = 2 * prec * rec / (prec + rec + 1e-6)
        
        # Average Recall (AR)
        ar = self._compute_average_recall()
        metrics['AR'] = ar
        
        return metrics
    
    def _compute_ap(self, iou_threshold: float) -> float:
        """Compute AP at given IoU threshold"""
        all_scores = []
        all_matches = []
        total_gt = 0
        
        for pred, tgt in zip(self.all_predictions, self.all_targets):
            pred_boxes = pred['boxes']
            pred_scores = pred['scores']
            gt_boxes = tgt['boxes']
            
            total_gt += len(gt_boxes)
            
            if len(pred_boxes) == 0:
                continue
            
            # Sort by score
            sorted_idx = torch.argsort(pred_scores, descending=True)
            
            matched_gt = set()
            
            for idx in sorted_idx:
                score = pred_scores[idx].item()
                all_scores.append(score)
                
                if len(gt_boxes) == 0:
                    all_matches.append(0)
                    continue
                
                # Compute IoU with all GT boxes
                pred_box = pred_boxes[idx:idx+1]
                ious = self._compute_iou(pred_box, gt_boxes).squeeze(0)
                
                if len(ious) > 0:
                    max_iou, max_idx = ious.max(0)
                    
                    if max_iou >= iou_threshold and max_idx.item() not in matched_gt:
                        all_matches.append(1)
                        matched_gt.add(max_idx.item())
                    else:
                        all_matches.append(0)
                else:
                    all_matches.append(0)
        
        if len(all_scores) == 0 or total_gt == 0:
            return 0.0
        
        # Sort by score
        sorted_indices = np.argsort(all_scores)[::-1]
        all_matches = np.array(all_matches)[sorted_indices]
        
        # Compute precision-recall curve
        tp_cumsum = np.cumsum(all_matches)
        fp_cumsum = np.cumsum(1 - all_matches)
        
        precisions = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-6)
        recalls = tp_cumsum / (total_gt + 1e-6)
        
        # 11-point interpolation AP
        ap = 0.0
        for t in np.linspace(0, 1, 11):
            mask = recalls >= t
            if mask.any():
                ap += precisions[mask].max() / 11
        
        return ap
    
    def _compute_precision_recall(self, iou_threshold: float) -> Tuple[float, float]:
        """Compute precision and recall at given IoU threshold"""
        total_tp = 0
        total_fp = 0
        total_gt = 0
        
        for pred, tgt in zip(self.all_predictions, self.all_targets):
            pred_boxes = pred['boxes']
            gt_boxes = tgt['boxes']
            
            total_gt += len(gt_boxes)
            
            if len(pred_boxes) == 0:
                continue
            
            if len(gt_boxes) == 0:
                total_fp += len(pred_boxes)
                continue
            
            # Match predictions to GT
            ious = self._compute_iou(pred_boxes, gt_boxes)  # (N, M)
            
            matched_gt = set()
            for i in range(len(pred_boxes)):
                if len(gt_boxes) == 0:
                    total_fp += 1
                    continue
                    
                max_iou, max_j = ious[i].max(0)
                
                if max_iou >= iou_threshold and max_j.item() not in matched_gt:
                    total_tp += 1
                    matched_gt.add(max_j.item())
                else:
                    total_fp += 1
        
        precision = total_tp / (total_tp + total_fp + 1e-6)
        recall = total_tp / (total_gt + 1e-6)
        
        return precision, recall
    
    def _compute_average_recall(self) -> float:
        """Compute Average Recall across IoU thresholds [0.5:0.95:0.05]"""
        recalls = []
        for iou_thresh in np.arange(0.5, 1.0, 0.05):
            _, rec = self._compute_precision_recall(iou_thresh)
            recalls.append(rec)
        return np.mean(recalls)
    
    def _compute_iou(self, boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
        """Compute IoU between two sets of boxes"""
        b1 = self._cxcywh_to_xyxy(boxes1)
        b2 = self._cxcywh_to_xyxy(boxes2)
        
        area1 = (b1[:, 2] - b1[:, 0]) * (b1[:, 3] - b1[:, 1])
        area2 = (b2[:, 2] - b2[:, 0]) * (b2[:, 3] - b2[:, 1])
        
        inter_x1 = torch.max(b1[:, None, 0], b2[None, :, 0])
        inter_y1 = torch.max(b1[:, None, 1], b2[None, :, 1])
        inter_x2 = torch.min(b1[:, None, 2], b2[None, :, 2])
        inter_y2 = torch.min(b1[:, None, 3], b2[None, :, 3])
        
        inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
        union = area1[:, None] + area2[None, :] - inter_area
        
        return inter_area / (union + 1e-6)
    
    def _cxcywh_to_xyxy(self, boxes: torch.Tensor) -> torch.Tensor:
        if len(boxes) == 0:
            return boxes
        cx, cy, w, h = boxes.unbind(-1)
        return torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=-1)


# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================
def train_one_epoch(model, criterion, dataloader, optimizer, device, epoch, config):
    """Train for one epoch"""
    model.train()
    
    total_losses = defaultdict(float)
    total_matched = 0
    num_batches = 0
    
    for batch_idx, (images, targets) in enumerate(dataloader):
        images = images.to(device)
        
        # Forward
        outputs = model(images)
        
        # Loss
        losses = criterion(outputs, targets)
        loss = losses['loss_total']
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.clip_grad)
        optimizer.step()
        
        # Accumulate
        for k, v in losses.items():
            if isinstance(v, torch.Tensor):
                total_losses[k] += v.item()
        total_matched += losses['num_matched'].item()
        num_batches += 1
        
        # Print progress
        if batch_idx % 100 == 0:
            avg_matched = total_matched / max(num_batches, 1)
            print(f"  Batch {batch_idx}/{len(dataloader)}, "
                  f"Loss: {loss.item():.4f}, "
                  f"Obj: {losses['loss_obj'].item():.4f}, "
                  f"BBox: {losses['loss_bbox'].item():.4f}, "
                  f"GIoU: {losses['loss_giou'].item():.4f}, "
                  f"Matched: {losses['num_matched'].item():.0f}, "
                  f"AvgMatched: {avg_matched:.1f}")
    
    avg_losses = {k: v / max(num_batches, 1) for k, v in total_losses.items()}
    avg_losses['avg_matched'] = total_matched / max(num_batches, 1)
    
    return avg_losses


@torch.no_grad()
def evaluate(model, criterion, dataloader, device, evaluator):
    """Evaluate model"""
    model.eval()
    evaluator.reset()
    
    total_losses = defaultdict(float)
    num_batches = 0
    
    for images, targets in dataloader:
        images = images.to(device)
        
        outputs = model(images)
        losses = criterion(outputs, targets)
        
        evaluator.update(outputs, targets)
        
        for k, v in losses.items():
            if isinstance(v, torch.Tensor):
                total_losses[k] += v.item()
        num_batches += 1
    
    avg_losses = {k: v / max(num_batches, 1) for k, v in total_losses.items()}
    metrics = evaluator.compute()
    
    return avg_losses, metrics


def save_checkpoint(model, optimizer, scheduler, epoch, train_losses, val_losses, 
                   val_metrics, best_metric, config, filename):
    """Save checkpoint"""
    model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
    
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model_state,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'val_metrics': val_metrics,
        'best_metric': best_metric,
        'config': config.__dict__
    }
    
    path = os.path.join(config.output_dir, filename)
    torch.save(checkpoint, path)
    return path


def train(config: Config):
    """Train from scratch"""
    print("=" * 70)
    print("PACR-DETR Detection Training")
    print("=" * 70)
    
    device = torch.device(config.device)
    print(f"Device: {device}")
    
    # Datasets
    print("\nLoading datasets...")
    train_dataset = SAVDetectionDataset(config, config.train_ann, is_train=True)
    val_dataset = SAVDetectionDataset(config, config.val_ann, is_train=False)
    
    print(f"  Train samples: {len(train_dataset)}")
    print(f"  Val samples: {len(val_dataset)}")
    
    if len(train_dataset) == 0:
        print("ERROR: No training samples!")
        return None, None
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        collate_fn=collate_fn,
        pin_memory=True
    )
    
    # Model
    print("\nBuilding model...")
    model = PACRDETRDetector(config)
    
    num_gpus = torch.cuda.device_count()
    if num_gpus > 1:
        print(f"  Using {num_gpus} GPUs with DataParallel")
        model = nn.DataParallel(model)
    
    model = model.to(device)
    
    num_params = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters: {num_params:,}")
    
    # Loss
    criterion = DetectionLoss(config).to(device)
    
    # Optimizer
    base_model = model.module if isinstance(model, nn.DataParallel) else model
    backbone_params = list(base_model.backbone.parameters())
    other_params = [p for n, p in base_model.named_parameters() if 'backbone' not in n]
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': config.lr_backbone},
        {'params': other_params, 'lr': config.lr}
    ], weight_decay=config.weight_decay)
    
    # Scheduler: Cosine annealing with warm restarts
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=10,
        T_mult=1,
        eta_min=1e-6
    )
    
    # Evaluator
    evaluator = DetectionEvaluator(config)
    
    history = {'train': [], 'val': []}
    best_mAP = 0.0
    
    print(f"\nStarting training for {config.epochs} epochs...")
    print("-" * 70)
    
    for epoch in range(1, config.epochs + 1):
        print(f"\nEpoch {epoch}/{config.epochs}")
        print(f"LR: backbone={optimizer.param_groups[0]['lr']:.2e}, head={optimizer.param_groups[1]['lr']:.2e}")
        
        # Train
        train_losses = train_one_epoch(model, criterion, train_loader, optimizer, device, epoch, config)
        
        # Evaluate
        print("\nEvaluating...")
        val_losses, val_metrics = evaluate(model, criterion, val_loader, device, evaluator)
        
        scheduler.step()
        
        # Print results
        print(f"\n{'='*70}")
        print(f"Epoch {epoch} Summary:")
        print(f"  Train Loss: {train_losses['loss_total']:.4f}, AvgMatched: {train_losses.get('avg_matched', 0):.1f}")
        print(f"  Val Loss: {val_losses['loss_total']:.4f}")
        print(f"  AP@50: {val_metrics['AP@50']:.4f}, AP@75: {val_metrics['AP@75']:.4f}, mAP: {val_metrics['mAP']:.4f}")
        print(f"  Precision: {val_metrics['precision']:.4f}, Recall: {val_metrics['recall']:.4f}, F1: {val_metrics['f1']:.4f}")
        print(f"  AR: {val_metrics['AR']:.4f}")
        
        # Save history
        history['train'].append(train_losses)
        history['val'].append({'losses': val_losses, 'metrics': val_metrics})
        
        # Best model
        is_best = val_metrics['mAP'] > best_mAP
        if is_best:
            best_mAP = val_metrics['mAP']
            print(f"  *** New best mAP: {best_mAP:.4f} ***")
            save_checkpoint(model, optimizer, scheduler, epoch, train_losses,
                          val_losses, val_metrics, best_mAP, config, 'best.pth')
        
        # Save latest
        save_checkpoint(model, optimizer, scheduler, epoch, train_losses,
                       val_losses, val_metrics, best_mAP, config, 'latest.pth')
        
        # Save periodic
        if epoch % config.save_every == 0:
            save_checkpoint(model, optimizer, scheduler, epoch, train_losses,
                          val_losses, val_metrics, best_mAP, config, f'epoch_{epoch}.pth')
            print(f"  Saved epoch_{epoch}.pth")
    
    print("\n" + "=" * 70)
    print(f"Training completed! Best mAP: {best_mAP:.4f}")
    print("=" * 70)
    
    return model, history


def resume_training(config: Config, checkpoint_path: str):
    """Resume training from checkpoint"""
    print("=" * 70)
    print("PACR-DETR Detection - Resume Training")
    print("=" * 70)
    
    device = torch.device(config.device)
    print(f"Device: {device}")
    
    # Load checkpoint
    print(f"\nLoading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    start_epoch = checkpoint['epoch'] + 1
    best_mAP = checkpoint.get('best_metric', 0.0)
    
    print(f"  Resuming from epoch {start_epoch}")
    print(f"  Previous best mAP: {best_mAP:.4f}")
    
    # Datasets
    print("\nLoading datasets...")
    train_dataset = SAVDetectionDataset(config, config.train_ann, is_train=True)
    val_dataset = SAVDetectionDataset(config, config.val_ann, is_train=False)
    
    print(f"  Train samples: {len(train_dataset)}")
    print(f"  Val samples: {len(val_dataset)}")
    
    if len(train_dataset) == 0:
        print("ERROR: No training samples!")
        return None, None
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        collate_fn=collate_fn,
        pin_memory=True,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        collate_fn=collate_fn,
        pin_memory=True
    )
    
    # Model
    print("\nBuilding model...")
    model = PACRDETRDetector(config)
    
    num_gpus = torch.cuda.device_count()
    if num_gpus > 1:
        print(f"  Using {num_gpus} GPUs with DataParallel")
        model = nn.DataParallel(model)
    
    model = model.to(device)
    
    # Load weights
    model_state = checkpoint['model_state_dict']
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(model_state)
    else:
        model.load_state_dict(model_state)
    print("  ✓ Model weights loaded")
    
    # Loss
    criterion = DetectionLoss(config).to(device)
    
    # Optimizer
    base_model = model.module if isinstance(model, nn.DataParallel) else model
    backbone_params = list(base_model.backbone.parameters())
    other_params = [p for n, p in base_model.named_parameters() if 'backbone' not in n]
    
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': config.lr_backbone},
        {'params': other_params, 'lr': config.lr}
    ], weight_decay=config.weight_decay)
    
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print("  ✓ Optimizer state loaded")
    
    # Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=10,
        T_mult=1,
        eta_min=1e-6
    )
    if checkpoint.get('scheduler_state_dict'):
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    print("  ✓ Scheduler state loaded")
    
    # Evaluator
    evaluator = DetectionEvaluator(config)
    
    history = {'train': [], 'val': []}
    
    print(f"\nResuming training from epoch {start_epoch} to {config.epochs}...")
    print("-" * 70)
    
    for epoch in range(start_epoch, config.epochs + 1):
        print(f"\nEpoch {epoch}/{config.epochs}")
        print(f"LR: backbone={optimizer.param_groups[0]['lr']:.2e}, head={optimizer.param_groups[1]['lr']:.2e}")
        
        # Train
        train_losses = train_one_epoch(model, criterion, train_loader, optimizer, device, epoch, config)
        
        # Evaluate
        print("\nEvaluating...")
        val_losses, val_metrics = evaluate(model, criterion, val_loader, device, evaluator)
        
        scheduler.step()
        
        # Print results
        print(f"\n{'='*70}")
        print(f"Epoch {epoch} Summary:")
        print(f"  Train Loss: {train_losses['loss_total']:.4f}, AvgMatched: {train_losses.get('avg_matched', 0):.1f}")
        print(f"  Val Loss: {val_losses['loss_total']:.4f}")
        print(f"  AP@50: {val_metrics['AP@50']:.4f}, AP@75: {val_metrics['AP@75']:.4f}, mAP: {val_metrics['mAP']:.4f}")
        print(f"  Precision: {val_metrics['precision']:.4f}, Recall: {val_metrics['recall']:.4f}, F1: {val_metrics['f1']:.4f}")
        print(f"  AR: {val_metrics['AR']:.4f}")
        
        # Save history
        history['train'].append(train_losses)
        history['val'].append({'losses': val_losses, 'metrics': val_metrics})
        
        # Best model
        is_best = val_metrics['mAP'] > best_mAP
        if is_best:
            best_mAP = val_metrics['mAP']
            print(f"  *** New best mAP: {best_mAP:.4f} ***")
            save_checkpoint(model, optimizer, scheduler, epoch, train_losses,
                          val_losses, val_metrics, best_mAP, config, 'best.pth')
        
        # Save latest
        save_checkpoint(model, optimizer, scheduler, epoch, train_losses,
                       val_losses, val_metrics, best_mAP, config, 'latest.pth')
        
        # Save periodic
        if epoch % config.save_every == 0:
            save_checkpoint(model, optimizer, scheduler, epoch, train_losses,
                          val_losses, val_metrics, best_mAP, config, f'epoch_{epoch}.pth')
            print(f"  Saved epoch_{epoch}.pth")
    
    print("\n" + "=" * 70)
    print(f"Training completed! Best mAP: {best_mAP:.4f}")
    print("=" * 70)
    
    return model, history


# ============================================================================
# INFERENCE FUNCTIONS
# ============================================================================
class PACRDETRInference:
    """
    Inference wrapper for PACR-DETR Detection
    
    Usage:
        inferencer = PACRDETRInference(checkpoint_path, config)
        boxes, scores = inferencer.detect(image)
        boxes, scores = inferencer.detect_batch(images)
    """
    
    def __init__(self, checkpoint_path: str, config: Config = None, device: str = None):
        self.config = config or Config()
        self.device = torch.device(device or self.config.device)
        
        # Load model
        self.model = PACRDETRDetector(self.config)
        
        checkpoint = torch.load(checkpoint_path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model = self.model.to(self.device)
        self.model.eval()
        
        print(f"✓ Loaded model from {checkpoint_path}")
        print(f"  Checkpoint epoch: {checkpoint.get('epoch', 'unknown')}")
        print(f"  Best mAP: {checkpoint.get('best_metric', 'unknown')}")
        
        # Transform
        self.transform = transforms.Compose([
            transforms.Resize((self.config.img_size, self.config.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    @torch.no_grad()
    def detect(self, image: Image.Image, conf_threshold: float = None, 
               return_format: str = 'xyxy') -> Tuple[np.ndarray, np.ndarray]:
        """
        Detect persons in single image
        
        Args:
            image: PIL Image
            conf_threshold: Confidence threshold (default: config.conf_threshold)
            return_format: 'xyxy' (x1,y1,x2,y2) or 'cxcywh' (center x, center y, w, h)
        
        Returns:
            boxes: (N, 4) numpy array, normalized [0,1]
            scores: (N,) numpy array, confidence scores
        """
        conf_threshold = conf_threshold or self.config.conf_threshold
        
        # Get original size
        orig_w, orig_h = image.size
        
        # Transform
        img_tensor = self.transform(image).unsqueeze(0).to(self.device)
        
        # Forward
        outputs = self.model(img_tensor)
        
        pred_obj = outputs['pred_obj'][0].sigmoid().squeeze(-1).cpu().numpy()  # (Q,)
        pred_boxes = outputs['pred_boxes'][0].cpu().numpy()  # (Q, 4)
        
        # Filter by confidence
        valid_mask = pred_obj > conf_threshold
        
        if valid_mask.sum() == 0:
            return np.zeros((0, 4)), np.zeros(0)
        
        boxes = pred_boxes[valid_mask]
        scores = pred_obj[valid_mask]
        
        # Convert format if needed
        if return_format == 'xyxy':
            boxes = self._cxcywh_to_xyxy(boxes)
        
        # Sort by score
        sorted_idx = np.argsort(scores)[::-1]
        boxes = boxes[sorted_idx]
        scores = scores[sorted_idx]
        
        return boxes, scores
    
    @torch.no_grad()
    def detect_batch(self, images: List[Image.Image], conf_threshold: float = None,
                     return_format: str = 'xyxy') -> List[Tuple[np.ndarray, np.ndarray]]:
        """
        Detect persons in batch of images
        
        Args:
            images: List of PIL Images
            conf_threshold: Confidence threshold
            return_format: 'xyxy' or 'cxcywh'
        
        Returns:
            List of (boxes, scores) tuples
        """
        conf_threshold = conf_threshold or self.config.conf_threshold
        
        # Transform all images
        img_tensors = torch.stack([self.transform(img) for img in images]).to(self.device)
        
        # Forward
        outputs = self.model(img_tensors)
        
        results = []
        
        B = len(images)
        for b in range(B):
            pred_obj = outputs['pred_obj'][b].sigmoid().squeeze(-1).cpu().numpy()
            pred_boxes = outputs['pred_boxes'][b].cpu().numpy()
            
            valid_mask = pred_obj > conf_threshold
            
            if valid_mask.sum() == 0:
                results.append((np.zeros((0, 4)), np.zeros(0)))
                continue
            
            boxes = pred_boxes[valid_mask]
            scores = pred_obj[valid_mask]
            
            if return_format == 'xyxy':
                boxes = self._cxcywh_to_xyxy(boxes)
            
            sorted_idx = np.argsort(scores)[::-1]
            results.append((boxes[sorted_idx], scores[sorted_idx]))
        
        return results
    
    @torch.no_grad()
    def detect_with_nms(self, image: Image.Image, conf_threshold: float = None,
                        nms_threshold: float = 0.5, return_format: str = 'xyxy') -> Tuple[np.ndarray, np.ndarray]:
        """
        Detect with Non-Maximum Suppression
        
        Args:
            image: PIL Image
            conf_threshold: Confidence threshold
            nms_threshold: IoU threshold for NMS
            return_format: 'xyxy' or 'cxcywh'
        
        Returns:
            boxes: (N, 4) after NMS
            scores: (N,)
        """
        conf_threshold = conf_threshold or self.config.conf_threshold
        
        # Get detections
        boxes, scores = self.detect(image, conf_threshold, return_format='xyxy')
        
        if len(boxes) == 0:
            if return_format == 'cxcywh':
                return np.zeros((0, 4)), np.zeros(0)
            return boxes, scores
        
        # Apply NMS
        keep_indices = self._nms(boxes, scores, nms_threshold)
        
        boxes = boxes[keep_indices]
        scores = scores[keep_indices]
        
        if return_format == 'cxcywh':
            boxes = self._xyxy_to_cxcywh(boxes)
        
        return boxes, scores
    
    def _cxcywh_to_xyxy(self, boxes: np.ndarray) -> np.ndarray:
        """Convert cxcywh to xyxy format"""
        if len(boxes) == 0:
            return boxes
        cx, cy, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
        x1 = cx - w / 2
        y1 = cy - h / 2
        x2 = cx + w / 2
        y2 = cy + h / 2
        return np.stack([x1, y1, x2, y2], axis=1)
    
    def _xyxy_to_cxcywh(self, boxes: np.ndarray) -> np.ndarray:
        """Convert xyxy to cxcywh format"""
        if len(boxes) == 0:
            return boxes
        x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        w = x2 - x1
        h = y2 - y1
        return np.stack([cx, cy, w, h], axis=1)
    
    def _nms(self, boxes: np.ndarray, scores: np.ndarray, threshold: float) -> np.ndarray:
        """Non-Maximum Suppression"""
        x1 = boxes[:, 0]
        y1 = boxes[:, 1]
        x2 = boxes[:, 2]
        y2 = boxes[:, 3]
        
        areas = (x2 - x1) * (y2 - y1)
        order = scores.argsort()[::-1]
        
        keep = []
        while order.size > 0:
            i = order[0]
            keep.append(i)
            
            xx1 = np.maximum(x1[i], x1[order[1:]])
            yy1 = np.maximum(y1[i], y1[order[1:]])
            xx2 = np.minimum(x2[i], x2[order[1:]])
            yy2 = np.minimum(y2[i], y2[order[1:]])
            
            w = np.maximum(0, xx2 - xx1)
            h = np.maximum(0, yy2 - yy1)
            inter = w * h
            
            iou = inter / (areas[i] + areas[order[1:]] - inter)
            
            inds = np.where(iou <= threshold)[0]
            order = order[inds + 1]
        
        return np.array(keep)
    
    def visualize(self, image: Image.Image, boxes: np.ndarray, scores: np.ndarray,
                  output_path: str = None) -> Image.Image:
        """
        Visualize detections on image
        
        Args:
            image: PIL Image
            boxes: (N, 4) in xyxy format, normalized [0,1]
            scores: (N,) confidence scores
            output_path: Optional path to save visualization
        
        Returns:
            PIL Image with drawn boxes
        """
        import PIL.ImageDraw as ImageDraw
        import PIL.ImageFont as ImageFont
        
        img = image.copy()
        draw = ImageDraw.Draw(img)
        
        w, h = img.size
        
        for i, (box, score) in enumerate(zip(boxes, scores)):
            # Convert normalized to pixel coordinates
            x1 = int(box[0] * w)
            y1 = int(box[1] * h)
            x2 = int(box[2] * w)
            y2 = int(box[3] * h)
            
            # Draw box
            color = (0, 255, 0)  # Green
            draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
            
            # Draw label
            label = f"P{i}: {score:.2f}"
            draw.text((x1, y1 - 15), label, fill=color)
        
        if output_path:
            img.save(output_path)
            print(f"Saved visualization to {output_path}")
        
        return img

class PACRDETRInferenceNoNMS:
    """
    Inference wrapper for PACR-DETR Detection (No NMS Version)

    Usage:
        inferencer = PACRDETRInferenceNoNMS(checkpoint_path, config)
        boxes, scores = inferencer.detect(image)
        boxes, scores = inferencer.detect_batch(images)
    """

    def __init__(self, checkpoint_path: str, config: Config = None, device: str = None):
        self.config = config or Config()
        self.device = torch.device(device or self.config.device)

        # Load model
        self.model = PACRDETRDetector(self.config)
        checkpoint = torch.load(checkpoint_path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model = self.model.to(self.device)
        self.model.eval()

        print(f"✓ Loaded model from {checkpoint_path}")
        print(f"  Checkpoint epoch: {checkpoint.get('epoch', 'unknown')}")
        print(f"  Best mAP: {checkpoint.get('best_metric', 'unknown')}")

        # Transform
        self.transform = transforms.Compose([
            transforms.Resize((self.config.img_size, self.config.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    @torch.no_grad()
    def detect(self, image: Image.Image, conf_threshold: float = None,
               return_format: str = 'xyxy') -> Tuple[np.ndarray, np.ndarray]:
        """
        Detect persons in single image (No NMS)

        Returns:
            boxes: (N, 4) numpy array, normalized [0,1]
            scores: (N,) numpy array
        """

        conf_threshold = conf_threshold or self.config.conf_threshold

        # Transform
        img_tensor = self.transform(image).unsqueeze(0).to(self.device)

        # Forward
        outputs = self.model(img_tensor)

        pred_obj = outputs['pred_obj'][0].sigmoid().squeeze(-1).cpu().numpy()
        pred_boxes = outputs['pred_boxes'][0].cpu().numpy()

        # Filter by confidence only
        valid_mask = pred_obj > conf_threshold

        if valid_mask.sum() == 0:
            return np.zeros((0, 4)), np.zeros(0)

        boxes = pred_boxes[valid_mask]
        scores = pred_obj[valid_mask]

        # Convert format if needed
        if return_format == 'xyxy':
            boxes = self._cxcywh_to_xyxy(boxes)

        # Sort by confidence
        sorted_idx = np.argsort(scores)[::-1]
        boxes = boxes[sorted_idx]
        scores = scores[sorted_idx]

        return boxes, scores

    @torch.no_grad()
    def detect_batch(self, images: List[Image.Image], conf_threshold: float = None,
                     return_format: str = 'xyxy') -> List[Tuple[np.ndarray, np.ndarray]]:
        """
        Batch detection (No NMS)
        """

        conf_threshold = conf_threshold or self.config.conf_threshold

        img_tensors = torch.stack(
            [self.transform(img) for img in images]
        ).to(self.device)

        outputs = self.model(img_tensors)

        results = []

        for b in range(len(images)):
            pred_obj = outputs['pred_obj'][b].sigmoid().squeeze(-1).cpu().numpy()
            pred_boxes = outputs['pred_boxes'][b].cpu().numpy()

            valid_mask = pred_obj > conf_threshold

            if valid_mask.sum() == 0:
                results.append((np.zeros((0, 4)), np.zeros(0)))
                continue

            boxes = pred_boxes[valid_mask]
            scores = pred_obj[valid_mask]

            if return_format == 'xyxy':
                boxes = self._cxcywh_to_xyxy(boxes)

            sorted_idx = np.argsort(scores)[::-1]
            results.append((boxes[sorted_idx], scores[sorted_idx]))

        return results

    def _cxcywh_to_xyxy(self, boxes: np.ndarray) -> np.ndarray:
        if len(boxes) == 0:
            return boxes

        cx, cy, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
        x1 = cx - w / 2
        y1 = cy - h / 2
        x2 = cx + w / 2
        y2 = cy + h / 2

        return np.stack([x1, y1, x2, y2], axis=1)

    def _xyxy_to_cxcywh(self, boxes: np.ndarray) -> np.ndarray:
        if len(boxes) == 0:
            return boxes

        x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        w = x2 - x1
        h = y2 - y1

        return np.stack([cx, cy, w, h], axis=1)

    def visualize(self, image: Image.Image, boxes: np.ndarray,
                  scores: np.ndarray, output_path: str = None) -> Image.Image:

        import PIL.ImageDraw as ImageDraw

        img = image.copy()
        draw = ImageDraw.Draw(img)
        w, h = img.size

        for box, score in zip(boxes, scores):
            x1 = int(box[0] * w)
            y1 = int(box[1] * h)
            x2 = int(box[2] * w)
            y2 = int(box[3] * h)

            color = (0, 255, 0)
            draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
            draw.text((x1, y1 - 15), f"{score:.2f}", fill=color)

        if output_path:
            img.save(output_path)
            print(f"Saved visualization to {output_path}")

        return img



# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================
def export_detections_for_tst(inferencer: PACRDETRInference, 
                              data_root: str,
                              clip_names: List[str],
                              output_file: str,
                              conf_threshold: float = 0.5):
    """
    Export detected bounding boxes for TST action classification
    
    Args:
        inferencer: PACRDETRInference instance
        data_root: Path to frames directory
        clip_names: List of clip names to process
        output_file: Output JSON file path
        conf_threshold: Confidence threshold
    
    Output format:
        {
            "clip_name": {
                "frame_idx": [
                    {"box": [x1, y1, x2, y2], "score": 0.95, "person_id": 0},
                    ...
                ],
                ...
            },
            ...
        }
    """
    data_root = Path(data_root)
    results = {}
    
    for clip_name in clip_names:
        clip_dir = data_root / clip_name
        if not clip_dir.exists():
            continue
        
        results[clip_name] = {}
        
        # Get all frames
        frames = sorted([f for f in clip_dir.iterdir() if f.suffix.lower() in ['.jpg', '.png']])
        
        for frame_idx, frame_path in enumerate(frames):
            try:
                image = Image.open(frame_path).convert('RGB')
                boxes, scores = inferencer.detect_with_nms(image, conf_threshold, nms_threshold=0.5)
                
                detections = []
                for i, (box, score) in enumerate(zip(boxes, scores)):
                    detections.append({
                        'box': box.tolist(),
                        'score': float(score),
                        'person_id': i
                    })
                
                results[clip_name][frame_idx] = detections
                
            except Exception as e:
                print(f"Error processing {frame_path}: {e}")
                continue
    
    # Save
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"✓ Exported detections to {output_file}")
    print(f"  Processed {len(results)} clips")
    
    return results


# ============================================================================
# MAIN
# ============================================================================
if __name__ == "__main__":
    # Set seeds
    torch.manual_seed(42)
    np.random.seed(42)
    random.seed(42)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
    
    # Create config
    config = Config()
    
    # ========================================
    # OPTION 1: Train from scratch
    # ========================================
    # model, history = train(config)
    
    # ========================================
    # OPTION 2: Resume training
    # ========================================
    # checkpoint_path = '/kaggle/input/models/YOUR_KAGGLE_USERNAME/pacr4/pytorch/default/1/best.pth'
    # config.epochs = 10
    # model, history = resume_training(config, checkpoint_path)
    
    # ========================================
    # OPTION 3: Inference only
    # ========================================
    # inferencer = PACRDETRInference('/kaggle/input/models/YOUR_KAGGLE_USERNAME/pacr4/pytorch/default/1/best.pth', config)
    # image = Image.open('/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/scb-dataset/SCB5-Handrise-Read-write-2024-9-17/images/val/0001003.jpg')
    # boxes, scores = inferencer.detect_with_nms(image, conf_threshold=0.5)
    # vis_img = inferencer.visualize(image, boxes, scores, 'output.jpg')

    #No NMS
    # inferencer = PACRDETRInferenceNoNMS(
    #     '/kaggle/input/models/YOUR_KAGGLE_USERNAME/pacr8/pytorch/default/1/best.pth',
    #     config
    # )

    # image = Image.open(
    #     '/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/my-dataset/5k_HRW_yolo_Dataset_jpg/images/train/0001076.jpg'
    # )

    # # No NMS — only confidence filtering
    # boxes, scores = inferencer.detect(image, conf_threshold=0.5)

    # vis_img = inferencer.visualize(image, boxes, scores, 'output.jpg')
    
    # ========================================
    # OPTION 4: Export for TST
    # ========================================
    inferencer = PACRDETRInference('/kaggle/input/models/YOUR_KAGGLE_USERNAME/pacr8/pytorch/default/1/best.pth', config)
    export_detections_for_tst(
        inferencer,
        config.data_root,
        clip_names=['20181016_05_1064', '20181016_05_1065'],
        output_file='detections.json'
    )
    
    # print("=" * 70)
    # print("PACR-DETR Detection Model Ready")
    # print("=" * 70)
    # print("\nUsage options:")
    # print("  1. Train from scratch: model, history = train(config)")
    # print("  2. Resume training: model, history = resume_training(config, checkpoint_path)")
    # print("  3. Inference: inferencer = PACRDETRInference(checkpoint_path)")
    # print("  4. Export for TST: export_detections_for_tst(inferencer, ...)")
    # print("=" * 70)

✓ Loaded pretrained ResNet50 weights
✓ Loaded model from /kaggle/input/models/YOUR_KAGGLE_USERNAME/pacr8/pytorch/default/1/best.pth
  Checkpoint epoch: 8
  Best mAP: 0.7687588908763449
✓ Exported detections to detections.json
  Processed 2 clips
